# Chapter 15: The Likelihood Ratio Test and Wilks' Theorem

The computational applications of this chapter of *Theory of Statistics*. The code is generated automatically from the book's `code/ch15/` files; see the book for the explanations and proofs.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ibrahimguney/statistical-theory/blob/main/notebooks/ch_15.ipynb)

$$
\newcommand{\R}{\mathbb{R}}\newcommand{\N}{\mathbb{N}}\newcommand{\E}{\mathbb{E}}
\newcommand{\P}{\mathbb{P}}\newcommand{\F}{\mathcal{F}}\newcommand{\I}{\mathbb{I}}
\newcommand{\Var}{\operatorname{Var}}\newcommand{\Cov}{\operatorname{Cov}}
\newcommand{\Corr}{\operatorname{Corr}}\newcommand{\MSE}{\operatorname{MSE}}
\newcommand{\Bias}{\operatorname{Bias}}\newcommand{\Normal}{\mathcal{N}}
\newcommand{\iid}{\overset{\text{i.i.d.}}{\sim}}\newcommand{\pto}{\xrightarrow{P}}
\newcommand{\dto}{\xrightarrow{d}}\newcommand{\X}{\mathbf{X}}\newcommand{\x}{\mathbf{x}}
$$

In [1]:
# Setup: numpy, scipy and matplotlib are preinstalled on Colab. Locally:
#   pip install -r code/requirements.txt
# Figure files (PDF) saved by the code are written to the _figures/ folder; the figures are
# also shown inside the notebook.
import os
os.makedirs("_figures", exist_ok=True)
os.chdir("_figures")
%matplotlib inline

## Verifying Wilks' Theorem (Poisson)

*Source file:* `code/ch15/c15_1_verifying_wilks_theorem_poisson.py`

In [2]:
import numpy as np
import scipy.stats as stats

rng = np.random.default_rng(42)
samples = rng.poisson(lam=4.0, size=(150_000, 100))
xb = np.mean(samples, axis=1)
g2 = 2.0 * 100 * (xb * np.log(xb / 4.0) - (xb - 4.0))
print(f"E[G^2]: {np.mean(g2):.4f} (theory: 1.0) | type I error: {np.mean(g2 > stats.chi2.ppf(0.95, 1)):.4f}")
print(f"Var[G^2]: {np.var(g2):.4f} (theory: 2.0) | KS p-value (against chi2_1): {stats.kstest(g2, 'chi2', args=(1,)).pvalue:.3f}")

E[G^2]: 0.9998 (theory: 1.0) | type I error: 0.0517
Var[G^2]: 2.0141 (theory: 2.0) | KS p-value (against chi2_1): 0.000


## The Holy Trinity: Exact Sizes in Small Samples

Since the Poisson sum $T=n\bar X\sim\text{Poisson}(n\lambda_0)$, the true sizes of the three tests under $H_0$ can be computed exactly, without simulation.

*Source file:* `code/ch15/c15_2_holy_trinity_exact_sizes.py`

In [3]:
import numpy as np
from scipy import stats

def sizes(n, lam0, alpha=0.05):
    t = np.arange(0, int(n*lam0 + 20*np.sqrt(n*lam0) + 30))
    w = stats.poisson.pmf(t, n*lam0)
    xb = t / n
    with np.errstate(divide="ignore", invalid="ignore"):
        G = 2*n*(np.where(xb > 0, xb*np.log(xb/lam0), 0.0) - (xb - lam0))
        W = np.where(xb > 0, n*(xb - lam0)**2 / xb, np.inf)
    S = n*(xb - lam0)**2 / lam0
    k = stats.chi2.ppf(1 - alpha, 1)
    return [np.sum(w[s > k]) for s in (G, W, S)]

print(" n   lambda0 |  LRT    Wald   Score  (target 0.05)")
for n, l0 in [(5, 1.0), (10, 1.0), (10, 0.3), (50, 1.0), (200, 1.0)]:
    g, w, s = sizes(n, l0)
    print(f"{n:4d}  {l0:5.1f}   | {g:.4f} {w:.4f} {s:.4f}")

 n   lambda0 |  LRT    Wald   Score  (target 0.05)
   5    1.0   | 0.0723 0.1301 0.0386
  10    1.0   | 0.0563 0.0743 0.0374
  10    0.3   | 0.0833 0.2030 0.0335
  50    1.0   | 0.0474 0.0512 0.0556
 200    1.0   | 0.0476 0.0537 0.0517


## Multinomial Goodness of Fit

*Source file:* `code/ch15/c15_3_multinomial_goodness_of_fit.py`

In [4]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(42)
p0 = np.array([0.1, 0.2, 0.3, 0.4])
n, R = 40, 200_000
N = rng.multinomial(n, p0, size=R)
E = n * p0
with np.errstate(divide="ignore", invalid="ignore"):
    G2 = 2 * np.sum(np.where(N > 0, N * np.log(N / E), 0.0), axis=1)
X2 = np.sum((N - E)**2 / E, axis=1)
k = stats.chi2.ppf(0.95, df=3)
print(f"n={n}: P(G^2 > chi2_3,0.05) = {np.mean(G2 > k):.4f} | P(X^2 > chi2_3,0.05) = {np.mean(X2 > k):.4f}")
print(f"means: G^2={G2.mean():.3f}, X^2={X2.mean():.3f} (theory 3) | Corr(G^2, X^2)={np.corrcoef(G2, X2)[0,1]:.4f}")
# An example data set
observed = np.array([7, 5, 13, 15])
r1 = stats.chisquare(observed, E)
r2 = stats.power_divergence(observed, E, lambda_="log-likelihood")
print(f"Observed {observed}, expected {E}")
print(f"Pearson X^2 = {r1.statistic:.4f}, p = {r1.pvalue:.4f} | G^2 = {r2.statistic:.4f}, p = {r2.pvalue:.4f}")

n=40: P(G^2 > chi2_3,0.05) = 0.0600 | P(X^2 > chi2_3,0.05) = 0.0474
means: G^2=3.104, X^2=2.996 (theory 3) | Corr(G^2, X^2)=0.9666
Observed [ 7  5 13 15], expected [ 4.  8. 12. 16.]
Pearson X^2 = 3.5208, p = 0.3181 | G^2 = 3.2795, p = 0.3505
